# Write your first ATST file. Example 1

Read the source data, define each ATST block, and write one self-contained ATST file.

## Setup

Import the ATST block classes, writer, device reader, and data utilities.

In [19]:
from pathlib import Path

import pandas as pd

from ATST import ATSTFile, write_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    Study,
)
from ATST import LogPhase600Reader


## Input data

Read the device export and layout. The reader supplies the measurements plus instrument metadata and assay information.

If you do not have a layout file, you can generate it at https://atstgenerator.streamlit.app/Layout_Editor using your wells configuration, or using any spreadsheet following the [specification](../../SPEC.md)

In [20]:
raw_path = Path("simulated_logphase_output.txt")
device_output = LogPhase600Reader().read(raw_path.read_bytes(), filename=raw_path.name)
layout_data = pd.read_csv("example_1_layout.tsv", sep="\t", keep_default_na=False)


## FILE_INFO

`FILE_INFO` identifies the output file and ATST format. The file name and creation date are required.

In [21]:
file_info = FileMetadata(data={
    "file_name": "example_1.atst.txt",
    "created_on": "2026-01-01",
    
})


## STUDY

`STUDY` identifies and describes the experiment. A non-empty title and study ID are required. Any other fields are optional.

In [22]:
study = Study(data={
    "title": "Example 1 single-readout in-file assay",
    "study_id": "ATST_EXAMPLE_1",
    "description": "Single readout study with metadata, assay, entities, layout, and readings declared in-file.",
})


## METADATA

`METADATA` records how the measurement was acquired. The block is required, but it has no mandatory fields.

In [23]:
display(device_output.metadata)

metadata = Metadata(data=device_output.metadata)


{'instrument': 'LogPhase600',
 'plate_type': 'Corning 96 well for LogPhase600',
 'date_start': '',
 'operator': 'simulated',
 'experiment_type': 'logphase_output'}

## ASSAY

`ASSAY` describes the measured signal. Readout type, readout unit, and time unit are required; a plate layout also requires plate format.

In [24]:
display(device_output.assay)

assay = Assay(data=device_output.assay)


{'readout_type': 'absorbance',
 'readout_unit': 'od600',
 'time_unit': 'seconds',
 'plate_format': '96_well',
 'read_wavelenth': '600nm',
 'filename': 'simulated_logphase_output',
 'plate_num': '1',
 'device_plate_number': '1',
 'runtime': '12h',
 'interval': '40min',
 'read_count': '19',
 'incubator': 'On',
 'temperature_setpoint': '37 C',
 'temperature_gradient': 'On',
 'shaking': 'On',
 'shaking_frequency': '100 RPMs'}

## ENTITIES

`ENTITIES` defines the biological records referenced by the layout. Each table declares a primary key with unique, non-empty values. These tables should be generated before constructing the final ATST file.

In [25]:
phages_table = pd.read_csv("entities/PHAGES.csv", keep_default_na=False)
display(phages_table)
isolates_table = pd.read_csv("entities/ISOLATES.csv", keep_default_na=False)
display(isolates_table)

phages = EntityTable(
    name="ENTITIES",
    table_name="PHAGES",
    pk="PHAGE_ID",
    data=phages_table,
)
isolates = EntityTable(
    name="ENTITIES",
    table_name="ISOLATES",
    pk="ISOLATE_ID",
    data=isolates_table,
)
entities = Entities(tables={"PHAGES": phages, "ISOLATES": isolates})


,PHAGE_ID,accession_number
0,ph_1,A00000000000001
1,ph_2,A00000000000002
2,ph_3,A00000000000003
3,ph_4,A00000000000004
4,ph_5,A00000000000005
5,ph_6,A00000000000006
6,ph_7,A00000000000007


,ISOLATE_ID
0,isolate_1
1,prop_host_1
2,prop_host_2
3,prop_host_3
4,PAO1


## LAYOUT

`LAYOUT` maps each readings column to its experimental and biological context. Its `well_loc` or `curve_id` key must be unique and match the readings columns.

In [26]:
display(layout_data.head())

layout = Layout(data=layout_data)


,well_loc,type,isolate_id,moi,phage_id
0,A1,treated,isolate_1,0.1,ph_1
1,A2,treated,isolate_1,0.1,ph_1
2,A3,treated,isolate_1,0.1,ph_1
3,A4,blank,,,
4,A5,treated,isolate_1,1,ph_1


## READINGS

`READINGS` contains the time series. `Time` is the first column and the remaining column names must match layout keys.

In [27]:
display(device_output.readings.head())

readings = Readings(data=device_output.readings)


,Time,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12
0,0,0.168,0.168,0.169,0.101,0.176,0.170,0.154,0.099,0.164,...,0.161,0.095,0.155,0.182,0.166,0.108,0.159,0.111,0.081,0.157
1,2400,0.174,0.169,0.206,0.104,0.181,0.193,0.186,0.114,0.168,...,0.188,0.109,0.174,0.206,0.180,0.091,0.182,0.084,0.105,0.178
2,4800,0.225,0.206,0.230,0.116,0.213,0.193,0.225,0.101,0.193,...,0.223,0.092,0.199,0.203,0.215,0.128,0.217,0.103,0.100,0.180
3,7200,0.283,0.300,0.264,0.079,0.301,0.291,0.305,0.092,0.195,...,0.280,0.094,0.274,0.278,0.280,0.095,0.287,0.110,0.100,0.234
4,9600,0.341,0.362,0.351,0.094,0.369,0.353,0.360,0.094,0.226,...,0.351,0.093,0.378,0.382,0.367,0.101,0.343,0.083,0.110,0.286


## Write the ATST file

Combine the blocks and write the complete single-file dataset.

In [28]:
atst = ATSTFile(
    file_info=file_info,
    study=study,
    metadata=metadata,
    assay=assay,
    entities=entities,
    layout=layout,
    readings=readings,
)
write_atst(atst, "example_1.atst.txt")


PosixPath('example_1.atst.txt')